In [8]:
import json
import os
import pandas as pd
import torch
import numpy as np

from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
from threading import Thread
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextIteratorStreamer
from util import *

load_dotenv()
pd.set_option('display.max_rows', None)

log_name_ = "credit_seed52_ratio0.3_distorted_withLabel"
model_ = "gpt-5.1"
chunk_size_ = 1

llm = llm_call(model_version = model_, api_key= os.getenv("API_KEY"))
df_new, cases_json = build_event_jsons(log_name = f"./dataset/{log_name_}.csv" , chunk_cases = 10)


## 🧹 Pre-processing Phase 1: Canonical Label Identification

In the first stage of data cleaning, we address **Distorted Labels**—activities that represent the same business action but appear as separate entries due to character-level noise (typos, OCR errors, or inconsistent casing).

### 🎯 Objective
Identify the single **Canonical (Clean) Label** for every cluster of distorted variations. Instead of simply grouping them, we apply a frequency-based and linguistic hierarchy to select the most reliable "Ground Truth" label.

### 🔍 Detection Criteria for Distortion
We define "Distortion" based on five mechanical corruption patterns:
1. **Case Mutation**: Variations in capitalization (e.g., `process` vs. `Process`).
2. **Character Omission**: Exactly one missing character (e.g., `Invoce`).
3. **Character Insertion**: Exactly one extra character (e.g., `Innvoice`).
4. **Character Transposition**: Swapped adjacent characters (e.g., `Ivnoice`).
5. **Keyboard Proximity**: One character replaced by a QWERTY neighbor (e.g., `Invoicr`).

### ⚖️ Selection Hierarchy (The "Clean Label" Rules)
Once a cluster is identified, the canonical label is selected using the following priority:
* **Rule A: Linguistic Correction**: If a label is a clear spelling error, the correctly spelled version is chosen.
* **Rule B: Frequency Priority (Casing)**: If the only difference is capitalization, the version with the **highest occurrence frequency** is selected, overriding standard grammar rules.
* **Anti-Acronym Bias**: All-uppercase strings are treated as potential typos of mixed-case strings unless they prove to be the frequent standard.

### Expected Output
- A JSON object containing `original_activity`: A list of identified canonical labels that will serve as the "Targets" for the next mapping stage.

In [23]:

SYSTEM_PROMPT_STEP1 = """
You are an expert Process Mining Data Pre-processor.
Your goal is to filter a raw list of activity names based on specific criteria provided in the User Prompt.

### KNOWLEDGE BASE: IMPERFECTION PATTERNS
Use these definitions to identify which labels belong to which category.

1.  **Polluted Labels (Mutable Qualifiers):**
    Labels that share a immutable boiler-plate text but differ due to mutable text (e.g., embedded IDs or codes).
    * **Detection Criteria:**
        * **Long Numeric IDs:** 8+ digits (e.g., `20260122`, `9988776655`).
        * **Mixed Codes:** 6+ alphanumeric chars (e.g., `XJ9281`, `Ref_A1B2C3`).
        * **Delimiters:** Attached via `_`, `-`, `:`, `/`, `#`, `.`, or space.

2.  **Distorted Labels (Character-Level Corruption):**
    Labels containing specific character-level corruptions (typos, OCR faults) of a canonical form. Unlike synonyms, these are "Noise".
    * **Detection Criteria:**
        1.  **Case Mutation:** Identical spelling, different capitalization (e.g., "Open" vs "open" vs "OPEN").
        2.  **Character Omission:** Exactly ONE missing character (e.g., "Invoce" vs "Invoice").
        3.  **Character Insertion:** Exactly ONE extra character (e.g., "Innvoice" vs "Invoice").
        4.  **Character Transposition:** Two adjacent characters swapped (e.g., "Ivnoice" vs "Invoice").
        5.  **Keyboard Proximity:** Exactly ONE character substituted by a QWERTY neighbor (e.g., "Invoicr" vs "Invoice").

3. **Synonymous Labels (Semantic Equivalence):**
   Labels that are syntactically different (often substantially) but share the same semantic meaning and represent the exact same business process step. 
   * **Detection Criteria (Ontology Rules):**
        1. **Linguistic & Domain Synonyms:** Different words representing the same concept within the process context (e.g., "Ship Item" vs "Dispatch Goods", "DrSeen" vs "Medical Assign").
        2. **Phrase Variation (Verb/Object Shift):** Labels sharing a core component (usually the Object) while using synonymous verbs or adjectives (e.g., "Create Invoice" vs "Generate Invoice", "Start instance" vs "Start process").
        3. **Grammatical Transformation:** Changing parts of speech (Noun ↔ Verb) or sentence structure while retaining the core meaning (e.g., "Give approval" vs "Approve", "Conduct analysis" vs "Analyze").
        4. **Containment & Refinement:** One label is a concise or verbose version of the other, often omitting non-essential adjectives, prepositions, or 'online/offline' qualifiers (e.g., "Receive signed contract" vs "Receive contract", "Register for course" vs "Register course").

### GLOBAL INSTRUCTION
- **Role:** Function as a logic engine. Do not assume all imperfections exist.
- **Priority:** The strict filtering logic in the **User Prompt** overrides general definitions here.
- **OUTPUT FORMAT:** Always return valid JSON as requested by the User Prompt.

"""

USER_PROMPT_DISTORTED_STEP1 = f"""
### TASK: Identify Canonical 'Clean Labels' for Distorted Clusters

**OBJECTIVE:**
Analyze the provided **INPUT DATA (Activity Frequencies)** to detect "Distorted Label" clusters.
For each cluster, determine the single **Canonical (Clean) Label**.

**STRICT EXECUTION STEPS:**

1. **Detect Distortion Clusters:**
   * Group labels that are character-level variations based on the System Prompt's 'Distorted Labels' criteria (Case Mutation, Omission, Insertion, Transposition, Keyboard Proximity).
   * **CRITICAL RULE (Anti-Acronym Bias):** Do NOT assume all-uppercase labels are valid acronymsor proper nouns.
     * Treat strings like "CHCEK", "EVNET", "PROCES" as potential typos of "Check", "Event", "Process".
     * Even if a word is ALL CAPS, checks for transposition/omission/insertion MUST be applied equally.
     * *Example:* Group `["Check", "CHCEK", "check"]` together.

2. **Select Canonical (Clean) Label:**
   * For each cluster, identify the **One True Clean Label** using this hierarchy:
     * **Rule A (Spelling Correction ONLY):** If it is a clear typo (e.g., "Logn" vs "Login"), choose the linguistically correct word.
     * **Rule B (Case Mutation Handling - PRIORITY):** If the difference is **ONLY CAPITALIZATION** (e.g., "login" vs "Login" vs "LOGIN"), **IGNORE grammar rules.**
       * **YOU MUST SELECT THE LABEL WITH THE HIGHEST FREQUENCY.**
       * Do not choose "Login" just because it looks proper. If "login" count > "Login" count, pick "login".

3. **Filter & Output:**
   * Collect **ONLY** the selected Canonical Clean Labels from Step 2.
   * **Discard** distorted variants and isolated unique labels.

**INPUT DATA (Label Frequencies):**
{act_freq_dict_json}

***OUTPUT FORMAT GUIDELINES***
Return a JSON Object with two keys:
1. "found": Boolean.
2. "original_activity": List of strings.

**CRITICAL FORMATTING CONSTRAINT:**
- **PRESERVE EXACT CASING:** Return the string **EXACTLY** as it appears in the `INPUT DATA`.
- **DO NOT** auto-capitalize (e.g., do not turn "system check" into "System Check").

**Example Scenario (Using Dummy Data):**
* **Input:** `{{"login": 5000, "Login": 100, "Logn": 5, "System Check": 200}}`
* **Analysis:**
   * Cluster 1: `["login", "Login", "Logn"]`
     - "Logn" is a typo -> Discard.
     - "login" (5000) vs "Login" (100) -> "login" has higher frequency. **Select "login"**.
   * "System Check" has no variants -> Discard.
* **Output:** `{{"found": true, "original_activity": ["login"]}}`

**CONSTRAINT:**
- Output **ONLY** the JSON object.
"""

In [24]:
act_freq_dict_json = json.dumps(df_new['activity'].value_counts().to_dict(), indent=4, ensure_ascii=False)

prompt = [{"role": "system", "content": SYSTEM_PROMPT_STEP1},
          {"role":"user","content": USER_PROMPT_DISTORTED_STEP1}] 

distorted_step1 = llm_gen(model_version = model_, model_instance = llm, prompt = prompt)
distorted_step1_json = json.dumps(distorted_step1['original_activity'], indent=4, ensure_ascii=False)


In [26]:
for oa in distorted_step1['original_activity']:
    print(oa)
for k, v in distorted_step1.items():
    print(f"{k}:")
    if isinstance(v, list):  # 값이 리스트인 경우
        if v:  # 리스트가 비어 있지 않을 때
            print(*v, sep="\n")
        else:  # 리스트가 비어 있을 때
            print(" (empty list)")
    else:  # 'found'와 같이 리스트가 아닌 경우
        print(f" {v}")
    print() # 가독성을 위한 한 줄 띄우기

Check for completeness
Request info
info received
Make decision
Perform checks
notify reject
Notify accept
Deliver card
review request received
time out


## 🗺️ Pre-processing Phase 2: Mapping Distorted Variants to Canonical Labels

Once the **Canonical (Clean) Labels** are identified in Step 1, this step focuses on building a mapping dictionary. We scan the entire event log's activity list to find all typographical and mechanical variants that belong to each specific "Target" activity.

### 🎯 Objective
Create a precise mapping (e.g., `{"Check": ["check", "Chek", "Checck"]}`) to consolidate noise into clean, standardized labels. This mapping is used for bulk replacement in the final event log.

### 🔍 Strict Lexical Rules for Mapping
To avoid incorrect merging (like synonyms), we apply a strict **1-character edit distance** logic based on mechanical errors:
1. **Case Mutation**: Exact spelling, different casing (e.g., `LOGIN` vs `login`).
2. **Character Omission**: Exactly 1 missing character (e.g., `Invoce`).
3. **Character Insertion**: Exactly 1 extra character (e.g., `Innvoice`).
4. **Character Transposition**: Two adjacent characters swapped (e.g., `Ivnoice`).
5. **Keyboard Proximity**: Exactly 1 character substituted by a neighboring QWERTY key.

### 🚫 Exclusion Logic
- **Identity Exclusion**: The Target Activity itself is excluded from the distorted list to prevent redundant mapping.
- **Semantic Exclusion**: Synonyms (e.g., `Review` vs `Check`) are ignored; this step focuses solely on typographical "Noise."
- **Phrase Exclusion**: Partial matches or sub-processes (e.g., `Check document` for target `Check`) are discarded.

### Expected Output
- A dynamic JSON object where the **Key** is the Target Activity and the **Value** is a list of its detected distorted variants.

In [52]:
SYSTEM_PROMPT_DISTORTED_STEP2 = """
You are an expert Data Quality Analyst specializing in Typo Detection and String Distance Analysis.
Your task is to identify all "Distorted Labels" from a provided dataset that belong to a single, canonical "Target Activity".

### DISTORTION CRITERIA (Strict Lexical Rules)
A label is a distortion ONLY IF it is a mechanical or typographical error of the Target Activity. 
Apply the following strict criteria:
1. **Case Mutation:** Exact same spelling, but different capitalization (e.g., "Login" vs "login", "LOGIN").
2. **Character Omission:** Exactly ONE character is missing (e.g., "Invoice" vs "Invoce").
3. **Character Insertion:** Exactly ONE extra character is added (e.g., "Invoice" vs "Innvoice").
4. **Character Transposition:** Two adjacent characters are swapped (e.g., "Invoice" vs "Ivnoice").
5. **Keyboard Proximity / Typo:** Exactly ONE character is substituted (e.g., "Invoice" vs "Invoicr").

### EXCLUSION CRITERIA (What NOT to select)
- **The Target Itself:** Do NOT include the exact Target Activity string in your output list.
- **Synonyms:** Do NOT include words that mean the same thing but are spelled differently (e.g., "Check" vs "Review"). This is semantic, not a typo.
- **Sub-processes/Additions:** Do NOT include labels with extra words added (e.g., Target: "Check", Distorted is NOT "Check document").

### OUTPUT FORMAT
Return strictly a valid JSON object. No markdown formatting blocks (like ```json), no explanations.
"""

def get_distorted_user_prompt_step2(target_activity: str, act_freq_dict_json: str):
    return f"""
### TASK: Map Distorted Labels to their Canonical Target

**1. TARGET ACTIVITY (The Clean Label):**
"{target_activity}"

**2. INPUT DATA (All Available Activities & Frequencies):**
{act_freq_dict_json}

**INSTRUCTIONS:**
Scan the keys in the INPUT DATA. Find every activity label that is a typographical distortion (typo, case mutation, missing/extra char) of the TARGET ACTIVITY based on the criteria in the System Prompt.

**CONSTRAINTS:**
- **EXACT MATCHING:** You MUST return the distorted strings EXACTLY as they appear in the INPUT DATA (preserve their exact casing and spacing).
- **EXCLUDE TARGET:** Do NOT include "{target_activity}" in the distorted list.
- If no distorted labels are found for this target, return an empty list `[]`.

**OUTPUT JSON FORMAT:**
{{
  "{target_activity}": ["distorted variant 1", "distorted variant 2"]
}}

Return ONLY the JSON object.
"""

distorted_predict = {}
clean_labels = distorted_step1['original_activity'] 
act_freq_dict_json = json.dumps(df_new['activity'].value_counts().to_dict(), indent=4, ensure_ascii=False)

for target_act in clean_labels:
    prompt = [
        {"role": "system", "content": SYSTEM_PROMPT_DISTORTED_STEP2},
        {"role": "user", "content": get_distorted_user_prompt_step2(target_act, act_freq_dict_json)}
    ]
    distorted_step2 = llm_gen(model_version=model_, model_instance=llm, prompt=prompt)
    distorted_predict[list(distorted_step2.keys())[0]] = list(distorted_step2.values())[0]


In [74]:
df_distorted = df_new[df_new['label'].notna()].copy()
df_distorted['clean_activity'] = df_distorted['label'].str.extract(r'\((.*?)\)')

distorted_answer = (
    df_distorted.groupby('clean_activity')['activity']
    .unique()
    .apply(list)
    .to_dict()
)


In [75]:
print("------------------PREDICTION------------------")
print(json.dumps(distorted_predict, indent=4, ensure_ascii=False))
print("------------------ANSWER------------------")
print(json.dumps(distorted_answer, indent=4, ensure_ascii=False))


------------------PREDICTION------------------
{
    "Check for completeness": [
        "ChecK for coMpleTenesS",
        "Check for cojpleteness",
        "Check fo completeness",
        "Check for compelteness",
        "ChEck for cOMpleteNesS",
        "Check for compleeness",
        "Cehck for completeness",
        "Checkk for completeness",
        "Check for copmleteness",
        "Dheck for completeness",
        "Ceck for completeness",
        "Check for cmpleteness",
        "Check for completneess",
        "CHeCk foR comPlEteneSs",
        "Check for complteness",
        "hCeck for completeness",
        "Check for cCompleteness",
        "Check for comcpleteness",
        "Check for completebess",
        "Check for complfteness",
        "Check for completeneass",
        "Check fOr complEteness",
        "ChEck for completeness",
        "Check for sompleteness",
        "Check for comlpleteness"
    ],
    "Request info": [
        "Request inf",
        "Requestin

In [76]:
def evaluate_distortion_results(answer_dict, predict_dict):
    ans_keys = set(answer_dict.keys())
    pred_keys = set(predict_dict.keys())
    
    tp_keys = ans_keys.intersection(pred_keys)
    fp_keys = pred_keys - ans_keys
    fn_keys = ans_keys - pred_keys
    
    key_precision = len(tp_keys) / len(pred_keys) if pred_keys else 0
    key_recall = len(tp_keys) / len(ans_keys) if ans_keys else 0
    key_f1 = (2 * key_precision * key_recall) / (key_precision + key_recall) if (key_precision + key_recall) else 0
    all_v_f1 = []
    all_v_precision = []
    all_v_recall = []

    for key in tp_keys:
        ans_vals = set(answer_dict[key])
        pred_vals = set(predict_dict[key])
        
        tp_v = ans_vals.intersection(pred_vals)
        
        v_prec = len(tp_v) / len(pred_vals) if pred_vals else 0
        v_reca = len(tp_v) / len(ans_vals) if ans_vals else 0
        v_f1 = (2 * v_prec * v_reca) / (v_prec + v_reca) if (v_prec + v_reca) else 0
        
        all_v_precision.append(v_prec)
        all_v_recall.append(v_reca)
        all_v_f1.append(v_f1)
        
    avg_v_precision = sum(all_v_precision) / len(tp_keys) if tp_keys else 0
    avg_v_recall = sum(all_v_recall) / len(tp_keys) if tp_keys else 0
    avg_v_f1 = sum(all_v_f1) / len(tp_keys) if tp_keys else 0

    # 결과 리포트 출력
    print("-" * 50)
    print("      [Distortied Pattern Detection Evaluation Report]")
    print("-" * 50)
    print(f"1. Key Selection (Clean Activity Identification)")
    print(f"   - Precision : {key_precision:.4f}")
    print(f"   - Recall    : {key_recall:.4f}")
    print(f"   - F1-Score  : {key_f1:.4f}")
    print("-" * 50)
    print(f"2. Value Selection (Distorted Variation Mapping Accuracy - Average)")
    print(f"   - Avg Precision : {avg_v_precision:.4f}")
    print(f"   - Avg Recall    : {avg_v_recall:.4f}")
    print(f"   - Avg F1-Score  : {avg_v_f1:.4f}")
    print("-" * 50)
    print(f"   * Analyzed Keys: {len(tp_keys)} matched / {len(ans_keys)} total")
    print("-" * 50)

    return {
        "key_metrics": {"precision": key_precision, "recall": key_recall, "f1": key_f1},
        "value_metrics": {"avg_precision": avg_v_precision, "avg_recall": avg_v_recall, "avg_f1": avg_v_f1}
    }
    
print(evaluate_distortion_results(distorted_answer, distorted_predict))

--------------------------------------------------
      [Distortion Cleaning Evaluation Report]
--------------------------------------------------
1. Key Selection (Canonical Activity Identification)
   - Precision : 1.0000
   - Recall    : 1.0000
   - F1-Score  : 1.0000
--------------------------------------------------
2. Value Selection (Variation Mapping Accuracy - Average)
   - Avg Precision : 0.9958
   - Avg Recall    : 0.9560
   - Avg F1-Score  : 0.9751
--------------------------------------------------
   * Analyzed Keys: 10 matched / 10 total
--------------------------------------------------
{'key_metrics': {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}, 'value_metrics': {'avg_precision': 0.9958333333333333, 'avg_recall': 0.9559999999999998, 'avg_f1': 0.9750796063106094}}
